In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Test FIWL on the simulated bank (EXP003)

Fisher Information Weighted Likelihood (FIWL) による CAT 実験。  
最初の問題は MFI 基準、2 問目以降は MLWI 基準で項目選択を行う。  
`catR` パッケージを使用して項目選択・能力推定を行い、結果を `EXP003/results/` に保存する。

In [ ]:
# --- Google Colab: R ランタイムで実行してください ---
# catR がインストールされていない場合のみインストール
if (!requireNamespace("catR", quietly = TRUE)) {
  install.packages("catR")
}
suppressPackageStartupMessages({
  library(catR)
  library(parallel)
})

In [ ]:
# --- プロジェクトルートの検出 ---
find_project_root <- function() {
  candidates <- c(
    getwd(),
    file.path(getwd(), "Grad_Research_new"),
    file.path(getwd(), "Grad_Research"),
    dirname(getwd()),
    "/content/Grad_Research_new",
    "/content/Grad_Research",
    "/content/drive/MyDrive/Grad_Research_new",
    "/content/drive/MyDrive/Grad_Research",
    "/content/drive/MyDrive/Colab Notebooks/Grad_Research"
  )
  for (root in candidates) {
    if (dir.exists(file.path(root, "data"))) return(normalizePath(root))
  }
  # Colab でドライブをマウントして再試行
  tryCatch(system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\""),
           error = function(e) NULL)
  for (root in candidates) {
    if (dir.exists(file.path(root, "data"))) return(normalizePath(root))
  }
  stop("Could not find the project root. Place the repository at MyDrive/Grad_Research_new or /content/Grad_Research_new.")
}

ROOT        <- find_project_root()
EXP003_DIR  <- file.path(ROOT, "EXP003")
RESULTS_DIR <- file.path(EXP003_DIR, "results")

cat(sprintf("Project root: %s\nResults dir : %s\n", ROOT, RESULTS_DIR))

In [ ]:
# --- 関数定義 ---

estimate_theta <- function(item_bank, item_ids, responses, current_theta, theta_range = c(-4, 4)) {
  if (all(responses == 1)) {
    return(current_theta + (max(item_bank[, "b"]) - current_theta) / 2)
  }
  if (all(responses == 0)) {
    return(current_theta + (min(item_bank[, "b"]) - current_theta) / 2)
  }
  administered <- item_bank[item_ids, , drop = FALSE]
  as.numeric(thetaEst(administered, responses, method = "ML", range = theta_range))
}

select_fiwl_item <- function(item_bank, theta_current, item_ids, responses) {
  if (length(item_ids) == 0) {
    # 1 問目は MFI で選択
    return(nextItem(
      item_bank,
      theta     = theta_current,
      criterion = "MFI",
      method    = "ML",
      range     = c(-4, 4)
    )$item)
  }
  # 2 問目以降は MLWI で選択
  nextItem(
    item_bank,
    theta   = theta_current,
    out     = item_ids,
    x       = responses,
    criterion = "MLWI",
    method  = "ML",
    range   = c(-4, 4),
    parInt  = c(-4, 4, 33)
  )$item
}

run_fiwl_cat <- function(item_bank, theta_true, test_length, seed) {
  testing_size <- length(theta_true)
  n_cores <- max(1L, detectCores() - 1L)
  cl <- makeCluster(n_cores)
  on.exit(stopCluster(cl), add = TRUE)

  clusterSetRNGStream(cl, seed)
  clusterExport(cl, c("item_bank", "theta_true", "test_length",
                       "select_fiwl_item", "estimate_theta"))
  clusterEvalQ(cl, suppressPackageStartupMessages(library(catR)))

  results <- parLapply(cl, seq_len(testing_size), function(user_id) {
    item_ids      <- integer(0)
    responses     <- integer(0)
    theta_current <- runif(1, -0.5, 0.5)
    user_records  <- vector("list", test_length)

    for (step in seq_len(test_length)) {
      selected      <- select_fiwl_item(item_bank, theta_current, item_ids, responses)
      response      <- as.integer(genPattern(theta_true[user_id], item_bank[selected, , drop = FALSE]))
      item_ids      <- c(item_ids, selected)
      responses     <- c(responses, response)
      theta_current <- estimate_theta(item_bank, item_ids, responses, theta_current)

      user_records[[step]] <- data.frame(
        userID     = user_id,
        step       = step,
        itemID     = selected,
        resp       = response,
        theta_true = theta_true[user_id],
        theta_est  = theta_current,
        bias       = theta_current - theta_true[user_id]
      )
    }
    do.call(rbind, user_records)
  })

  do.call(rbind, results)
}

summarize_steps <- function(records) {
  by_step <- split(records, records$step)
  summary <- lapply(by_step, function(x) {
    r <- if (sd(x$theta_true) == 0 || sd(x$theta_est) == 0) {
      NA_real_
    } else {
      cor(x$theta_true, x$theta_est)
    }
    data.frame(
      step = x$step[1],
      Bias = mean(x$bias),
      RMSE = sqrt(mean(x$bias^2)),
      MAE  = mean(abs(x$bias)),
      r    = r
    )
  })
  do.call(rbind, summary)
}

In [ ]:
# --- パラメータ設定 ---
bank_type     <- "uncor"    # "uncor" | "cor"
bank_id       <- 1L
test_length   <- 40L
testing_size  <- 0L         # 0 = 全データ使用
n_items       <- 200L
seed          <- 20260430L
output_suffix <- ""
theta_csv     <- ""         # 外部 theta CSV パス (空なら theta_true を使用)

In [ ]:
# --- データ読み込み ---
bank_dir <- switch(
  bank_type,
  uncor = file.path(ROOT, "data", "uncorrelated_banks"),
  cor   = file.path(ROOT, "data", "correlated_banks"),
  stop("bank_type must be 'uncor' or 'cor'.")
)

item_bank_path <- file.path(bank_dir, sprintf("item_bank_%s_%d.csv", bank_type, bank_id))
item_bank <- as.matrix(read.csv(item_bank_path)[, c("a", "b", "c")])
item_bank <- item_bank[seq_len(min(n_items, nrow(item_bank))), , drop = FALSE]
item_bank <- cbind(item_bank, d = 1)

if (nchar(theta_csv) > 0) {
  theta_true <- read.csv(theta_csv)[["x"]]
  message("Loaded theta from: ", theta_csv, " (n=", length(theta_true), ")")
} else {
  theta_path <- file.path(ROOT, "data", "theta_true", sprintf("theta_true_%d.csv", bank_id))
  theta_true <- read.csv(theta_path)[["x"]]
  if (testing_size > 0) {
    theta_true <- theta_true[seq_len(min(testing_size, length(theta_true)))]
  }
}

stopifnot(test_length <= nrow(item_bank))

cat(sprintf("item bank : %d items x %d params\ntheta_true: %d examinees\n",
            nrow(item_bank), ncol(item_bank), length(theta_true)))

In [ ]:
# --- FIWL CAT 実行 ---
records <- run_fiwl_cat(item_bank, theta_true, test_length, seed)
summary_by_step <- summarize_steps(records)

print(tail(summary_by_step, 5))

In [ ]:
# --- 結果保存 ---
dir.create(RESULTS_DIR, recursive = TRUE, showWarnings = FALSE)

records_path <- file.path(RESULTS_DIR, sprintf("records_%s_%d_FIWL%s.csv", bank_type, bank_id, output_suffix))
summary_path <- file.path(RESULTS_DIR, sprintf("summary_%s_%d_FIWL%s.csv", bank_type, bank_id, output_suffix))

write.csv(records, records_path, row.names = FALSE)
write.csv(summary_by_step, summary_path, row.names = FALSE)

cat(sprintf("Saved records to: %s\nSaved summary to: %s\n", records_path, summary_path))